# EMR Analytics Showcase

This notebook presents an EMR-style analytics workflow using SQLite, SQL, pandas, and Jupyter Notebook.
It follows a portfolio-friendly structure similar to the Abalone notebook: clear sections, focused outputs, and practical findings.


## Goals

- Connect to the SQLite EMR dataset with a path that works from different notebook launch locations.
- Inspect the healthcare-style schema and load the core tables needed for analysis.
- Explore patient, appointment, billing, treatment-note, and lab-result activity.
- Build a concise set of charts and tables that are easy to discuss in a portfolio or interview.
- Finish with practical summary points about operational and clinical insights.


In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


def find_project_root() -> Path:
    search_bases = [Path.cwd(), Path.cwd() / 'notebooks']
    for base in search_bases:
        for candidate in [base, *base.parents]:
            if (candidate / 'data' / 'emr_showcase.db').exists():
                return candidate
    raise FileNotFoundError('Could not locate data/emr_showcase.db from the current working directory.')


PROJECT_ROOT = find_project_root()
DB_PATH = PROJECT_ROOT / 'data' / 'emr_showcase.db'
IMAGES_DIR = PROJECT_ROOT / 'images'
IMAGES_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Database path: {DB_PATH}')

conn = sqlite3.connect(DB_PATH)


## Load Data


In [ ]:
table_counts = pd.DataFrame(
    {
        'table': ['patients', 'appointments', 'treatment_notes', 'payments', 'lab_results'],
        'rows': [
            pd.read_sql_query('SELECT COUNT(*) AS n FROM patients', conn).iloc[0, 0],
            pd.read_sql_query('SELECT COUNT(*) AS n FROM appointments', conn).iloc[0, 0],
            pd.read_sql_query('SELECT COUNT(*) AS n FROM treatment_notes', conn).iloc[0, 0],
            pd.read_sql_query('SELECT COUNT(*) AS n FROM payments', conn).iloc[0, 0],
            pd.read_sql_query('SELECT COUNT(*) AS n FROM lab_results', conn).iloc[0, 0],
        ],
    }
)

display(table_counts)


In [ ]:
schema = pd.read_sql_query(
    """
    SELECT name, sql
    FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """,
    conn,
)

schema[['name']]


## Exploratory Data Analysis


In [ ]:
patients = pd.read_sql_query('SELECT * FROM patients', conn)
patients.head()


In [ ]:
plt.figure(figsize=(10, 6))
ax = patients['risk_level'].value_counts().reindex(['Low', 'Medium', 'High']).plot(
    kind='bar',
    color=['#43a047', '#fbc02d', '#e53935']
)
ax.set_title('Patients by Risk Level')
ax.set_xlabel('Risk Level')
ax.set_ylabel('Patient Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'emr_risk_levels.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
monthly_appointments = pd.read_sql_query(
    """
    SELECT strftime('%Y-%m', start_time) AS month,
           appointment_type,
           COUNT(*) AS appointment_count
    FROM appointments
    GROUP BY strftime('%Y-%m', start_time), appointment_type
    ORDER BY month, appointment_type
    """,
    conn,
)

monthly_appointments.head()


In [ ]:
pivot_appointments = monthly_appointments.pivot(
    index='month',
    columns='appointment_type',
    values='appointment_count'
).fillna(0)

pivot_appointments.plot(figsize=(12, 6), marker='o')
plt.title('Monthly Appointment Volume by Type')
plt.xlabel('Month')
plt.ylabel('Appointment Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'emr_monthly_appointments.png', dpi=150, bbox_inches='tight')
plt.show()


## Revenue And Billing


In [ ]:
revenue = pd.read_sql_query(
    """
    SELECT payment_status,
           ROUND(SUM(amount), 2) AS total_amount,
           COUNT(*) AS invoice_count
    FROM payments
    GROUP BY payment_status
    ORDER BY total_amount DESC
    """,
    conn,
)

display(revenue)


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=revenue, x='payment_status', y='total_amount', palette='Blues_d')
plt.title('Revenue by Payment Status')
plt.xlabel('Payment Status')
plt.ylabel('Total Amount')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'emr_revenue_by_status.png', dpi=150, bbox_inches='tight')
plt.show()


## Treatment Notes And Diagnosis Mix


In [ ]:
diagnosis_mix = pd.read_sql_query(
    """
    SELECT diagnosis_group,
           ROUND(AVG(note_length), 1) AS avg_note_length,
           COUNT(*) AS note_count
    FROM treatment_notes
    GROUP BY diagnosis_group
    ORDER BY note_count DESC
    """,
    conn,
)

display(diagnosis_mix)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=diagnosis_mix, x='diagnosis_group', y='note_count', palette='viridis')
plt.title('Treatment Notes by Diagnosis Group')
plt.xlabel('Diagnosis Group')
plt.ylabel('Note Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'emr_diagnosis_mix.png', dpi=150, bbox_inches='tight')
plt.show()


## Abnormal Lab Results


In [ ]:
abnormal_labs = pd.read_sql_query(
    """
    SELECT p.patient_code,
           p.full_name,
           p.age,
           l.test_name,
           l.result_value,
           l.result_flag,
           l.test_date
    FROM lab_results l
    JOIN patients p ON p.id = l.patient_id
    WHERE l.result_flag = 'Abnormal'
    ORDER BY l.test_date DESC
    LIMIT 15
    """,
    conn,
)

abnormal_labs


## Patient Utilisation


In [ ]:
patient_utilization = pd.read_sql_query(
    """
    SELECT
        p.patient_code,
        p.full_name,
        COALESCE(a.total_appointments, 0) AS total_appointments,
        COALESCE(t.total_notes, 0) AS total_notes,
        COALESCE(pay.paid_revenue, 0) AS paid_revenue
    FROM patients p
    LEFT JOIN (
        SELECT patient_id, COUNT(*) AS total_appointments
        FROM appointments
        GROUP BY patient_id
    ) a ON a.patient_id = p.id
    LEFT JOIN (
        SELECT patient_id, COUNT(*) AS total_notes
        FROM treatment_notes
        GROUP BY patient_id
    ) t ON t.patient_id = p.id
    LEFT JOIN (
        SELECT patient_id, ROUND(SUM(amount), 2) AS paid_revenue
        FROM payments
        WHERE payment_status = 'Paid'
        GROUP BY patient_id
    ) pay ON pay.patient_id = p.id
    ORDER BY paid_revenue DESC, total_appointments DESC
    LIMIT 10
    """,
    conn,
)

patient_utilization


## Summary

- The dataset combines clinical and operational tables in a way that supports realistic SQL analysis.
- Appointment activity, billing status, and diagnosis-group note volumes are all easy to inspect from the same SQLite source.
- The utilisation query is built with separate aggregates to avoid duplicate-counting revenue.
- The notebook saves reusable figures into `images/`, which also makes the project easier to document in GitHub.
- This format is well suited to a portfolio because it shows data access, business interpretation, and presentation in one place.


In [ ]:
conn.close()
